## Part 3.1: Create, Load, and Validate Data Quality Rules

### Objective

Create or load the persistent data-quality rules used to evaluate the contents of the Cyclistic source files.

Part 2 verified the structural schema of the source data. Part 3 evaluates whether the values inside that structure are complete, valid, consistent, and suitable for further processing.

This section defines the quality contract only. It does not scan source files or calculate quality metrics.

### Approved Schema Dependency

Data-quality rules must be associated with an approved schema contract.

Before the rules are initialised, this section verifies that:

* `active_schema_status` is `APPROVED`;
* `active_schema` is not empty;
* `active_schema_version` is populated;
* every enabled quality rule references columns available in the approved schema.

If the active schema has not been approved, the quality audit must not continue.

### Quality Rule Structure

The quality rules are stored in `quality_rules.json`.

Each rule contains:

* a unique rule ID;
* a category;
* the affected column or columns;
* a severity;
* an enabled status;
* rule-specific parameters;
* a human-readable description.

Supported severity levels are:

```text
WARNING
FAIL
```

A warning identifies a suspicious condition that does not automatically invalidate the row.

A failure identifies a violation that may cause the row to become a quarantine candidate during later processing.

### Initial Quality Contract

The initial rules cover:

* required values;
* blank identifiers;
* duplicate ride IDs;
* permitted categorical values;
* datetime completeness and ordering;
* ride-duration validity;
* missing station information;
* latitude and longitude ranges.

The initial decisions are:

```text
Missing or blank ride_id                    → FAIL
Duplicate ride_id across the dataset        → FAIL
Invalid member_casual value                 → FAIL
Missing started_at or ended_at              → FAIL
ended_at not later than started_at          → FAIL
Ride duration greater than 24 hours         → WARNING
Missing station names or IDs                → WARNING
Coordinates outside global geographic range → FAIL
```

Missing station information is treated as a warning because some ride types may not begin or end at a formal station.

### File-Management Policy

This section follows the same conservative persistence policy used by the schema profiler:

* create `quality_rules.json` when it does not exist;
* preserve an existing valid file;
* add missing top-level sections from defaults;
* add missing default rules without replacing existing customised rules;
* preserve invalid JSON files for human review;
* use in-memory defaults when an existing file cannot be parsed;
* write safe migrations atomically;
* record all creations, migrations, warnings, and failures in `pipeline_audit.log`.

### Expected Outcome

* The approved schema prerequisite is verified.
* `quality_rules.json` exists or safe in-memory defaults are available.
* Existing valid customised rules are preserved.
* Missing required rules are added safely.
* Rule IDs, severities, columns, and parameters are validated.
* Enabled rules reference columns available in the approved schema.
* `quality_rules` is ready for metric generation and assessment in later Part 3 stages.


In [ ]:
# ==========================================
# Part 3.1: Create, Load, and Validate
# Data Quality Rules
# ==========================================

import json
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Any


# ------------------------------------------
# Validate Part 3.1 dependencies
# ------------------------------------------

REQUIRED_PART_3_1_VARIABLES = {
    "QUALITY_RULES_PATH": QUALITY_RULES_PATH,
    "MASTER_SCHEMA_PATH": MASTER_SCHEMA_PATH,
    "MELBOURNE_TIMEZONE": MELBOURNE_TIMEZONE,
}

for variable_name, variable_value in (
    REQUIRED_PART_3_1_VARIABLES.items()
):
    if variable_value is None:
        raise RuntimeError(
            f"Required Part 3.1 variable is unavailable: "
            f"{variable_name}"
        )


if not isinstance(QUALITY_RULES_PATH, Path):
    raise TypeError(
        "QUALITY_RULES_PATH must be a pathlib.Path object."
    )

if not isinstance(MASTER_SCHEMA_PATH, Path):
    raise TypeError(
        "MASTER_SCHEMA_PATH must be a pathlib.Path object."
    )


# ------------------------------------------
# Reload master registry from disk
# ------------------------------------------

# Reloading ensures that manual schema approval changes made after
# Part 2 are reflected in the current notebook runtime.
try:
    with MASTER_SCHEMA_PATH.open(
        mode="r",
        encoding="utf-8",
    ) as master_schema_file:
        approved_master_registry = json.load(
            master_schema_file
        )

except json.JSONDecodeError as error:
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_FAILED: "
            "master_schema.json contains invalid JSON. "
            f"Error={error}"
        ),
        level="CRITICAL",
    )

    raise ValueError(
        "master_schema.json contains invalid JSON."
    ) from error


if not isinstance(approved_master_registry, dict):
    raise TypeError(
        "master_schema.json must contain a JSON object."
    )


# ------------------------------------------
# Validate approved schema prerequisite
# ------------------------------------------

ACTIVE_SCHEMA = approved_master_registry.get(
    "active_schema",
    {},
)

ACTIVE_SCHEMA_VERSION = approved_master_registry.get(
    "active_schema_version"
)

ACTIVE_SCHEMA_STATUS = approved_master_registry.get(
    "active_schema_status"
)


if ACTIVE_SCHEMA_STATUS != "APPROVED":
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            f"active_schema_status={ACTIVE_SCHEMA_STATUS}; "
            "expected=APPROVED."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires an approved active schema."
    )


if not isinstance(ACTIVE_SCHEMA, dict) or not ACTIVE_SCHEMA:
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            "active_schema is empty or invalid."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires a non-empty approved active schema."
    )


if (
    not isinstance(ACTIVE_SCHEMA_VERSION, str)
    or not ACTIVE_SCHEMA_VERSION.strip()
):
    log_event(
        message=(
            "QUALITY_RULE_INITIALISATION_BLOCKED: "
            "active_schema_version is missing or invalid."
        ),
        level="CRITICAL",
    )

    raise RuntimeError(
        "Part 3 requires a valid active_schema_version."
    )


# ------------------------------------------
# Define default quality rules
# ------------------------------------------

DEFAULT_QUALITY_RULES = {
    "metadata": {
        "ruleset_version": "1.0.0",
        "ruleset_status": "ACTIVE",
        "source_schema_version": ACTIVE_SCHEMA_VERSION,
        "created_at": None,
        "updated_at": None,
        "description": (
            "Cyclistic source-data quality contract."
        ),
    },
    "rules": {
        "DQ001_RIDE_ID_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "ride_id must not be null."
            ),
        },
        "DQ002_RIDE_ID_NOT_BLANK": {
            "category": "COMPLETENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_BLANK",
            "parameters": {},
            "description": (
                "ride_id must not contain an empty or "
                "whitespace-only value."
            ),
        },
        "DQ003_RIDE_ID_UNIQUE": {
            "category": "UNIQUENESS",
            "columns": ["ride_id"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "UNIQUE",
            "parameters": {
                "scope": "ALL_FILES",
            },
            "description": (
                "ride_id must be unique across all source files."
            ),
        },
        "DQ004_RIDEABLE_TYPE_ALLOWED": {
            "category": "VALIDITY",
            "columns": ["rideable_type"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "ALLOWED_VALUES",
            "parameters": {
                "allowed_values": [
                    "classic_bike",
                    "electric_bike",
                    "docked_bike",
                ],
                "case_sensitive": True,
            },
            "description": (
                "rideable_type must contain an approved "
                "Cyclistic bike category."
            ),
        },
        "DQ005_STARTED_AT_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["started_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "started_at must not be null."
            ),
        },
        "DQ006_ENDED_AT_REQUIRED": {
            "category": "COMPLETENESS",
            "columns": ["ended_at"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "ended_at must not be null."
            ),
        },
        "DQ007_END_AFTER_START": {
            "category": "CONSISTENCY",
            "columns": [
                "started_at",
                "ended_at",
            ],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "DATETIME_ORDER",
            "parameters": {
                "operator": "GREATER_THAN",
            },
            "description": (
                "ended_at must be later than started_at."
            ),
        },
        "DQ008_DURATION_MAXIMUM": {
            "category": "REASONABLENESS",
            "columns": [
                "started_at",
                "ended_at",
            ],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "DURATION_MAXIMUM",
            "parameters": {
                "maximum_hours": 24,
            },
            "description": (
                "Ride duration greater than 24 hours requires "
                "review."
            ),
        },
        "DQ009_MEMBER_TYPE_ALLOWED": {
            "category": "VALIDITY",
            "columns": ["member_casual"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "ALLOWED_VALUES",
            "parameters": {
                "allowed_values": [
                    "member",
                    "casual",
                ],
                "case_sensitive": True,
            },
            "description": (
                "member_casual must be either member or casual."
            ),
        },
        "DQ010_START_STATION_NAME_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_station_name"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start station names require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ011_START_STATION_ID_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["start_station_id"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing start station IDs require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ012_END_STATION_NAME_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_station_name"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end station names require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ013_END_STATION_ID_MISSING": {
            "category": "COMPLETENESS",
            "columns": ["end_station_id"],
            "severity": "WARNING",
            "enabled": True,
            "rule_type": "NOT_NULL",
            "parameters": {},
            "description": (
                "Missing end station IDs require review but "
                "may be valid for some ride types."
            ),
        },
        "DQ014_START_LAT_RANGE": {
            "category": "VALIDITY",
            "columns": ["start_lat"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -90,
                "maximum": 90,
                "allow_null": False,
            },
            "description": (
                "start_lat must be between -90 and 90."
            ),
        },
        "DQ015_END_LAT_RANGE": {
            "category": "VALIDITY",
            "columns": ["end_lat"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -90,
                "maximum": 90,
                "allow_null": False,
            },
            "description": (
                "end_lat must be between -90 and 90."
            ),
        },
        "DQ016_START_LNG_RANGE": {
            "category": "VALIDITY",
            "columns": ["start_lng"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -180,
                "maximum": 180,
                "allow_null": False,
            },
            "description": (
                "start_lng must be between -180 and 180."
            ),
        },
        "DQ017_END_LNG_RANGE": {
            "category": "VALIDITY",
            "columns": ["end_lng"],
            "severity": "FAIL",
            "enabled": True,
            "rule_type": "NUMERIC_RANGE",
            "parameters": {
                "minimum": -180,
                "maximum": 180,
                "allow_null": False,
            },
            "description": (
                "end_lng must be between -180 and 180."
            ),
        },
    },
    "run_history": [],
}


# ------------------------------------------
# Define required top-level sections
# ------------------------------------------

QUALITY_RULE_SECTION_TYPES = {
    "metadata": dict,
    "rules": dict,
    "run_history": list,
}


REQUIRED_RULE_FIELDS = {
    "category": str,
    "columns": list,
    "severity": str,
    "enabled": bool,
    "rule_type": str,
    "parameters": dict,
    "description": str,
}


SUPPORTED_QUALITY_SEVERITIES = {
    "WARNING",
    "FAIL",
}


# ------------------------------------------
# Set default metadata timestamps
# ------------------------------------------

quality_rules_initialisation_time = datetime.now(
    MELBOURNE_TIMEZONE
).isoformat(timespec="seconds")

DEFAULT_QUALITY_RULES["metadata"][
    "created_at"
] = quality_rules_initialisation_time

DEFAULT_QUALITY_RULES["metadata"][
    "updated_at"
] = quality_rules_initialisation_time


# ------------------------------------------
# Create quality rules file if missing
# ------------------------------------------

def create_quality_rules_if_missing(
    file_path: Path,
) -> bool:
    """
    Create quality_rules.json only when it does not already exist.
    """
    if file_path.exists():

        if not file_path.is_file():
            raise FileExistsError(
                "QUALITY_RULES_PATH exists but is not a file:\n"
                f"{file_path}"
            )

        log_event(
            message=(
                "QUALITY_RULES_FOUND: Existing quality rules "
                "will be preserved."
            ),
            level="INFO",
        )

        return False

    save_json_atomically(
        file_path=file_path,
        data=deepcopy(DEFAULT_QUALITY_RULES),
    )

    if not file_path.exists():
        raise OSError(
            "quality_rules.json was not created:\n"
            f"{file_path}"
        )

    log_event(
        message=(
            "QUALITY_RULES_CREATED: quality_rules.json was "
            "created using the default quality contract."
        ),
        level="INFO",
    )

    return True


# ------------------------------------------
# Load quality rules conservatively
# ------------------------------------------

def load_quality_rules(
    file_path: Path,
) -> tuple[dict, bool]:
    """
    Load quality_rules.json while preserving malformed files.

    Returns:
        rules_data
        loaded_from_disk
    """
    try:
        with file_path.open(
            mode="r",
            encoding="utf-8",
        ) as rules_file:
            loaded_rules = json.load(rules_file)

    except json.JSONDecodeError as error:
        log_event(
            message=(
                "QUALITY_RULES_INVALID_JSON: "
                "The existing file was preserved and in-memory "
                f"defaults will be used. Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    except OSError as error:
        log_event(
            message=(
                "QUALITY_RULES_READ_FAILED: "
                "In-memory defaults will be used. "
                f"Error={error}"
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    if not isinstance(loaded_rules, dict):
        log_event(
            message=(
                "QUALITY_RULES_INVALID_STRUCTURE: "
                "quality_rules.json must contain a JSON object. "
                "The file was preserved and defaults will be used "
                "in memory."
            ),
            level="CRITICAL",
        )

        return deepcopy(DEFAULT_QUALITY_RULES), False

    return loaded_rules, True


# ------------------------------------------
# Validate top-level quality-rule structure
# ------------------------------------------

def validate_quality_rule_structure(
    rules_data: dict,
) -> tuple[dict, list[str], list[str]]:
    """
    Add missing top-level sections safely.

    Invalid section types are replaced in memory only.
    """
    validated_rules = deepcopy(rules_data)

    missing_sections = []
    invalid_sections = []

    for section_name, expected_type in (
        QUALITY_RULE_SECTION_TYPES.items()
    ):
        if section_name not in validated_rules:
            validated_rules[section_name] = deepcopy(
                DEFAULT_QUALITY_RULES[section_name]
            )

            missing_sections.append(section_name)
            continue

        if not isinstance(
            validated_rules[section_name],
            expected_type,
        ):
            validated_rules[section_name] = deepcopy(
                DEFAULT_QUALITY_RULES[section_name]
            )

            invalid_sections.append(section_name)

    return validated_rules, missing_sections, invalid_sections


# ------------------------------------------
# Add missing default quality rules
# ------------------------------------------

def migrate_missing_quality_rules(
    rules_data: dict,
) -> list[str]:
    """
    Add missing default rules while preserving existing customised
    rules with the same rule IDs.
    """
    missing_rule_ids = []

    configured_rules = rules_data["rules"]

    for rule_id, default_rule in (
        DEFAULT_QUALITY_RULES["rules"].items()
    ):
        if rule_id not in configured_rules:
            configured_rules[rule_id] = deepcopy(
                default_rule
            )

            missing_rule_ids.append(rule_id)

    return missing_rule_ids


# ------------------------------------------
# Validate individual quality rules
# ------------------------------------------

def validate_quality_rule_definitions(
    rules_data: dict,
    approved_schema: dict,
) -> list[dict]:
    """
    Validate quality-rule fields, severity values, and referenced
    approved-schema columns.
    """
    validation_issues = []

    for rule_id, rule_definition in (
        rules_data["rules"].items()
    ):
        if not isinstance(rule_id, str) or not rule_id.strip():
            validation_issues.append(
                {
                    "rule_id": str(rule_id),
                    "issue": "INVALID_RULE_ID",
                }
            )
            continue

        if not isinstance(rule_definition, dict):
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "RULE_NOT_OBJECT",
                }
            )
            continue

        for field_name, expected_type in (
            REQUIRED_RULE_FIELDS.items()
        ):
            if field_name not in rule_definition:
                validation_issues.append(
                    {
                        "rule_id": rule_id,
                        "issue": "MISSING_FIELD",
                        "field": field_name,
                    }
                )
                continue

            if not isinstance(
                rule_definition[field_name],
                expected_type,
            ):
                validation_issues.append(
                    {
                        "rule_id": rule_id,
                        "issue": "INVALID_FIELD_TYPE",
                        "field": field_name,
                    }
                )

        severity = rule_definition.get("severity")

        if (
            isinstance(severity, str)
            and severity not in SUPPORTED_QUALITY_SEVERITIES
        ):
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": "UNSUPPORTED_SEVERITY",
                    "observed": severity,
                }
            )

        enabled = rule_definition.get("enabled")

        if enabled is not True:
            continue

        rule_columns = rule_definition.get(
            "columns",
            [],
        )

        if not isinstance(rule_columns, list):
            continue

        missing_schema_columns = [
            column_name
            for column_name in rule_columns
            if column_name not in approved_schema
        ]

        if missing_schema_columns:
            validation_issues.append(
                {
                    "rule_id": rule_id,
                    "issue": (
                        "COLUMN_NOT_IN_APPROVED_SCHEMA"
                    ),
                    "columns": missing_schema_columns,
                }
            )

    return validation_issues


# ------------------------------------------
# Initialise quality rules
# ------------------------------------------

quality_rules_created = (
    create_quality_rules_if_missing(
        file_path=QUALITY_RULES_PATH,
    )
)

quality_rules_raw, quality_rules_loaded_from_disk = (
    load_quality_rules(
        file_path=QUALITY_RULES_PATH,
    )
)


(
    quality_rules,
    quality_rules_missing_sections,
    quality_rules_invalid_sections,
) = validate_quality_rule_structure(
    rules_data=quality_rules_raw,
)


missing_quality_rule_ids = []

if "rules" not in quality_rules_invalid_sections:
    missing_quality_rule_ids = (
        migrate_missing_quality_rules(
            rules_data=quality_rules,
        )
    )


quality_rule_validation_issues = (
    validate_quality_rule_definitions(
        rules_data=quality_rules,
        approved_schema=ACTIVE_SCHEMA,
    )
)


# ------------------------------------------
# Synchronise ruleset schema version
# ------------------------------------------

stored_source_schema_version = (
    quality_rules["metadata"].get(
        "source_schema_version"
    )
)

if stored_source_schema_version != ACTIVE_SCHEMA_VERSION:
    log_event(
        message=(
            "QUALITY_RULE_SCHEMA_VERSION_UPDATED: "
            f"previous={stored_source_schema_version}; "
            f"current={ACTIVE_SCHEMA_VERSION}."
        ),
        level="WARNING",
    )

    quality_rules["metadata"][
        "source_schema_version"
    ] = ACTIVE_SCHEMA_VERSION

    quality_rules["metadata"][
        "updated_at"
    ] = datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds")


# ------------------------------------------
# Persist safe migrations
# ------------------------------------------

safe_quality_rule_migration_required = (
    quality_rules_loaded_from_disk
    and (
        bool(quality_rules_missing_sections)
        or bool(missing_quality_rule_ids)
        or stored_source_schema_version
        != ACTIVE_SCHEMA_VERSION
    )
    and not quality_rules_invalid_sections
    and not quality_rule_validation_issues
)


if safe_quality_rule_migration_required:
    save_json_atomically(
        file_path=QUALITY_RULES_PATH,
        data=quality_rules,
    )

    log_event(
        message=(
            "QUALITY_RULES_MIGRATION_SAVED: Missing sections, "
            "rules, or schema-version metadata were updated."
        ),
        level="WARNING",
    )


# ------------------------------------------
# Report validation issues
# ------------------------------------------

for issue in quality_rule_validation_issues:
    log_event(
        message=(
            "QUALITY_RULE_VALIDATION_ISSUE: "
            f"{issue}"
        ),
        level="ERROR",
    )


QUALITY_RULES_READY = (
    not quality_rules_invalid_sections
    and not quality_rule_validation_issues
)


if QUALITY_RULES_READY:
    log_event(
        message=(
            "QUALITY_RULES_READY: "
            f"rules={len(quality_rules['rules'])}; "
            f"schema_version={ACTIVE_SCHEMA_VERSION}."
        ),
        level="INFO",
    )


# ------------------------------------------
# Display quality-rule summary
# ------------------------------------------

enabled_quality_rules = {
    rule_id: rule_definition
    for rule_id, rule_definition in (
        quality_rules["rules"].items()
    )
    if rule_definition.get("enabled") is True
}


warning_rule_count = sum(
    1
    for rule in enabled_quality_rules.values()
    if rule.get("severity") == "WARNING"
)

failure_rule_count = sum(
    1
    for rule in enabled_quality_rules.values()
    if rule.get("severity") == "FAIL"
)


print("\n========== Data Quality Rule Initialisation ==========\n")

print(f"Quality rules path       : {QUALITY_RULES_PATH}")
print(f"Quality rules created    : {quality_rules_created}")
print(f"Ruleset version          : {quality_rules['metadata'].get('ruleset_version')}")
print(f"Ruleset status           : {quality_rules['metadata'].get('ruleset_status')}")
print(f"Source schema version    : {quality_rules['metadata'].get('source_schema_version')}")
print(f"Total rules              : {len(quality_rules['rules'])}")
print(f"Enabled rules            : {len(enabled_quality_rules)}")
print(f"Failure rules            : {failure_rule_count}")
print(f"Warning rules            : {warning_rule_count}")
print(f"Validation issues        : {len(quality_rule_validation_issues)}")

print("\nEnabled quality rules:")

for rule_id, rule_definition in (
    enabled_quality_rules.items()
):
    print(
        f"  {rule_id:<30} "
        f"{rule_definition['severity']:<8} "
        f"{rule_definition['rule_type']}"
    )


print("\n" + "=" * 70)

if QUALITY_RULES_READY:
    print("✓ Approved schema prerequisite verified.")
    print("✓ Quality rules created, loaded, or migrated.")
    print("✓ Enabled rules reference approved schema columns.")
    print("✓ Rules are ready for Part 3.2 metric generation.")
else:
    print("✗ Quality rules require correction before Part 3.2.")

print("=" * 70)

## Part 3.2: Generate File-Level and Dataset-Level Quality Metrics

### Objective

Scan every discovered CSV file and generate the measurements required for the data-quality assessment without loading the complete dataset into memory.

This section collects evidence only. It does not determine whether a quality rule passes, warns, or fails. Rule assessment will occur in Part 3.3.

### Full-File Scanning Policy

Unlike schema profiling, data-quality auditing must inspect the complete source dataset.

Sample-based scanning is not sufficient for reliably detecting:

* duplicate ride IDs;
* missing values;
* invalid category values;
* datetime inconsistencies;
* duration anomalies;
* coordinate violations;
* row-level quarantine candidates.

Each CSV is therefore read completely in configurable chunks.

The default chunk size is:

```text
100,000 rows
```

Only one chunk is held in memory at a time.

### Memory-Control Strategy

The audit keeps compact information in memory:

* row and column counts;
* null and blank counts;
* category frequencies;
* datetime and duration metrics;
* coordinate metrics;
* station-field metrics;
* bounded examples of affected rows;
* file-level and dataset-level summaries.

The audit does not retain:

* complete DataFrames;
* one combined DataFrame;
* every affected row as a Python object;
* every ride ID in memory.

### Duplicate Tracking

Duplicate `ride_id` detection must operate across chunks and files.

A temporary SQLite database is used to store:

```text
ride_id
source_file
source_row_number
```

After all files have been scanned, SQLite is queried to identify:

* duplicates within an individual file;
* duplicates across multiple files;
* total duplicate ride IDs;
* total rows affected by duplication.

The temporary database is removed after the required metrics and bounded evidence have been extracted.

### Source Lineage

Evidence records contain:

* source filename;
* source row number;
* ride ID when available;
* observed value or relevant context.

The evidence collection is bounded to prevent common issues from generating millions of in-memory records.

The default evidence limit is:

```text
100 examples per metric
```

Counts always represent the complete scan, even when only a limited number of examples are retained.

### Metrics Generated

For each file, this section measures:

* rows scanned;
* columns available;
* missing approved-schema columns;
* null counts by column;
* blank-string counts by column;
* observed `rideable_type` values;
* observed `member_casual` values;
* invalid or unparseable timestamps;
* non-positive ride durations;
* rides longer than 24 hours;
* minimum, maximum, and average ride duration;
* missing station names and IDs;
* non-numeric latitude and longitude values;
* coordinates below or above global geographic ranges;
* duplicate ride IDs within the file.

Dataset-level metrics include:

* total rows scanned;
* total files scanned;
* aggregate null and blank counts;
* combined category frequencies;
* aggregate datetime, duration, station, and coordinate metrics;
* duplicate ride IDs across the complete dataset.

### Failure Handling

Each file is processed independently.

If one file cannot be scanned:

* the failure is recorded;
* scanning continues with the remaining files;
* partial metrics are marked clearly;
* the file failure is not automatically classified as a quality-rule failure.

### In-Memory Outputs

This section produces:

```text
quality_metrics_by_file
dataset_quality_metrics
quality_metric_evidence
quality_scan_failures
```

These results remain in memory.

Part 3.3 will compare them with `quality_rules.json` and assign rule outcomes.

### Expected Outcome

* All discoverable source files are scanned in chunks.
* Full-dataset quality metrics are generated without combining all data in memory.
* Cross-file duplicate ride IDs are identified through disk-backed tracking.
* Evidence samples remain bounded.
* File-processing failures are recorded separately.
* Metrics are ready for quality-rule assessment in Part 3.3.


In [ ]:
# ==========================================
# Part 3.2: Generate File-Level and
# Dataset-Level Data Quality Metrics
# ==========================================

import sqlite3
from collections import Counter
from copy import deepcopy
from datetime import datetime
from pathlib import Path

import pandas as pd


# ------------------------------------------
# Runtime configuration
# ------------------------------------------

# These are optional config values. Defaults are used when the
# settings have not been added to config.json.
DQ_CHUNK_SIZE = int(
    config.get("DQ_CHUNK_SIZE", 100_000)
)

DQ_EVIDENCE_LIMIT = int(
    config.get("DQ_EVIDENCE_LIMIT", 100)
)

DQ_DUPLICATE_BATCH_SIZE = int(
    config.get("DQ_DUPLICATE_BATCH_SIZE", 10_000)
)


if DQ_CHUNK_SIZE <= 0:
    raise ValueError(
        "DQ_CHUNK_SIZE must be greater than zero."
    )

if DQ_EVIDENCE_LIMIT < 0:
    raise ValueError(
        "DQ_EVIDENCE_LIMIT cannot be negative."
    )

if DQ_DUPLICATE_BATCH_SIZE <= 0:
    raise ValueError(
        "DQ_DUPLICATE_BATCH_SIZE must be greater than zero."
    )


# ------------------------------------------
# Validate Part 3.2 dependencies
# ------------------------------------------

if not QUALITY_RULES_READY:
    raise RuntimeError(
        "Part 3.2 cannot continue because quality rules "
        "are not ready."
    )

if not discovered_files:
    raise RuntimeError(
        "Part 3.2 requires discovered_files from Part 2.4."
    )

if not ACTIVE_SCHEMA:
    raise RuntimeError(
        "Part 3.2 requires a non-empty approved schema."
    )


# ------------------------------------------
# Determine required audit columns
# ------------------------------------------

ENABLED_QUALITY_RULES = {
    rule_id: rule_definition
    for rule_id, rule_definition in quality_rules["rules"].items()
    if rule_definition.get("enabled") is True
}


QUALITY_REQUIRED_COLUMNS = sorted(
    {
        column_name
        for rule_definition in ENABLED_QUALITY_RULES.values()
        for column_name in rule_definition.get("columns", [])
    }
)


KNOWN_TEXT_COLUMNS = {
    "ride_id",
    "rideable_type",
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
    "member_casual",
}


KNOWN_DATETIME_COLUMNS = {
    "started_at",
    "ended_at",
}


COORDINATE_LIMITS = {
    "start_lat": (-90.0, 90.0),
    "end_lat": (-90.0, 90.0),
    "start_lng": (-180.0, 180.0),
    "end_lng": (-180.0, 180.0),
}


STATION_COLUMNS = {
    "start_station_name",
    "start_station_id",
    "end_station_name",
    "end_station_id",
}


CATEGORY_COLUMNS = {
    "rideable_type",
    "member_casual",
}


# ------------------------------------------
# Initialise output collections
# ------------------------------------------

quality_metrics_by_file = {}
quality_metric_evidence = {}
quality_scan_failures = []

dataset_quality_metrics = {
    "session_id": SESSION_ID,
    "started_at": datetime.now(
        MELBOURNE_TIMEZONE
    ).isoformat(timespec="seconds"),
    "completed_at": None,
    "scan_mode": "FULL_FILE_CHUNKED",
    "chunk_size": DQ_CHUNK_SIZE,
    "evidence_limit_per_metric": DQ_EVIDENCE_LIMIT,
    "files_discovered": len(discovered_files),
    "files_scanned_successfully": 0,
    "files_failed": 0,
    "total_rows_scanned": 0,
    "null_counts": Counter(),
    "blank_counts": Counter(),
    "category_counts": {
        "rideable_type": Counter(),
        "member_casual": Counter(),
    },
    "datetime_metrics": {
        "started_at_unparseable": 0,
        "ended_at_unparseable": 0,
        "non_positive_duration": 0,
        "duration_over_24_hours": 0,
        "valid_duration_count": 0,
        "duration_total_seconds": 0.0,
        "duration_min_seconds": None,
        "duration_max_seconds": None,
    },
    "station_missing_counts": Counter(),
    "coordinate_metrics": {
        column_name: {
            "null_count": 0,
            "non_numeric_count": 0,
            "below_minimum_count": 0,
            "above_maximum_count": 0,
        }
        for column_name in COORDINATE_LIMITS
    },
    "duplicates": {
        "duplicate_ride_id_count": 0,
        "duplicate_rows_affected": 0,
        "within_file_duplicate_ride_id_count": 0,
        "cross_file_duplicate_ride_id_count": 0,
    },
}


# ------------------------------------------
# Bounded evidence collection
# ------------------------------------------

def add_quality_evidence(
    metric_name: str,
    evidence_record: dict,
) -> None:
    """
    Retain only a bounded number of examples for each metric.

    Counts are calculated separately and are not affected by the
    evidence limit.
    """
    if DQ_EVIDENCE_LIMIT == 0:
        return

    evidence_collection = quality_metric_evidence.setdefault(
        metric_name,
        [],
    )

    if len(evidence_collection) < DQ_EVIDENCE_LIMIT:
        evidence_collection.append(
            deepcopy(evidence_record)
        )


# ------------------------------------------
# Counter conversion for JSON compatibility
# ------------------------------------------

def counter_to_dict(
    value,
):
    """
    Convert Counter objects recursively into standard dictionaries.
    """
    if isinstance(value, Counter):
        return dict(value)

    if isinstance(value, dict):
        return {
            key: counter_to_dict(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            counter_to_dict(item)
            for item in value
        ]

    return value


# ------------------------------------------
# Update duration aggregates
# ------------------------------------------

def update_duration_aggregates(
    metrics: dict,
    duration_seconds: pd.Series,
) -> None:
    """
    Update duration counts and summary statistics from valid
    numeric duration values.
    """
    valid_duration = duration_seconds.dropna()

    if valid_duration.empty:
        return

    valid_count = int(valid_duration.shape[0])
    duration_sum = float(valid_duration.sum())
    duration_min = float(valid_duration.min())
    duration_max = float(valid_duration.max())

    metrics["valid_duration_count"] += valid_count
    metrics["duration_total_seconds"] += duration_sum

    current_min = metrics["duration_min_seconds"]

    if current_min is None or duration_min < current_min:
        metrics["duration_min_seconds"] = duration_min

    current_max = metrics["duration_max_seconds"]

    if current_max is None or duration_max > current_max:
        metrics["duration_max_seconds"] = duration_max


# ------------------------------------------
# Create temporary duplicate tracker
# ------------------------------------------

DUPLICATE_TRACKER_PATH = Path(
    f"/content/cyclistic_dq_{SESSION_ID}.sqlite"
)

if DUPLICATE_TRACKER_PATH.exists():
    DUPLICATE_TRACKER_PATH.unlink()


duplicate_connection = sqlite3.connect(
    DUPLICATE_TRACKER_PATH
)

duplicate_cursor = duplicate_connection.cursor()

duplicate_cursor.execute(
    """
    CREATE TABLE ride_id_occurrences (
        ride_id TEXT NOT NULL,
        source_file TEXT NOT NULL,
        source_row_number INTEGER NOT NULL
    )
    """
)

duplicate_connection.commit()


# ------------------------------------------
# Insert ride IDs into SQLite
# ------------------------------------------

def insert_ride_id_occurrences(
    ride_ids: pd.Series,
    source_file: str,
    source_row_numbers: pd.Series,
) -> None:
    """
    Add non-null and non-blank ride IDs to the disk-backed
    duplicate tracker.
    """
    valid_records = []

    for ride_id, row_number in zip(
        ride_ids,
        source_row_numbers,
    ):
        if pd.isna(ride_id):
            continue

        cleaned_ride_id = str(ride_id).strip()

        if not cleaned_ride_id:
            continue

        valid_records.append(
            (
                cleaned_ride_id,
                source_file,
                int(row_number),
            )
        )

        if len(valid_records) >= DQ_DUPLICATE_BATCH_SIZE:
            duplicate_cursor.executemany(
                """
                INSERT INTO ride_id_occurrences (
                    ride_id,
                    source_file,
                    source_row_number
                )
                VALUES (?, ?, ?)
                """,
                valid_records,
            )

            valid_records.clear()

    if valid_records:
        duplicate_cursor.executemany(
            """
            INSERT INTO ride_id_occurrences (
                ride_id,
                source_file,
                source_row_number
            )
            VALUES (?, ?, ?)
            """,
            valid_records,
        )


# ------------------------------------------
# Create empty file-level metric structure
# ------------------------------------------

def initialise_file_metrics(
    file_name: str,
    available_columns: list[str],
    missing_columns: list[str],
) -> dict:
    """
    Create the file-level metrics structure used during chunked
    aggregation.
    """
    return {
        "session_id": SESSION_ID,
        "file_name": file_name,
        "scan_started_at": datetime.now(
            MELBOURNE_TIMEZONE
        ).isoformat(timespec="seconds"),
        "scan_completed_at": None,
        "status": "IN_PROGRESS",
        "rows_scanned": 0,
        "chunks_processed": 0,
        "available_columns": available_columns,
        "missing_quality_columns": missing_columns,
        "null_counts": Counter(),
        "blank_counts": Counter(),
        "category_counts": {
            "rideable_type": Counter(),
            "member_casual": Counter(),
        },
        "datetime_metrics": {
            "started_at_unparseable": 0,
            "ended_at_unparseable": 0,
            "non_positive_duration": 0,
            "duration_over_24_hours": 0,
            "valid_duration_count": 0,
            "duration_total_seconds": 0.0,
            "duration_min_seconds": None,
            "duration_max_seconds": None,
        },
        "station_missing_counts": Counter(),
        "coordinate_metrics": {
            column_name: {
                "null_count": 0,
                "non_numeric_count": 0,
                "below_minimum_count": 0,
                "above_maximum_count": 0,
            }
            for column_name in COORDINATE_LIMITS
        },
        "duplicates": {
            "duplicate_ride_id_count": 0,
            "duplicate_rows_affected": 0,
        },
    }


# ------------------------------------------
# Scan source files in chunks
# ------------------------------------------

log_event(
    message=(
        "DATA_QUALITY_METRIC_SCAN_START: "
        f"files={len(discovered_files)}; "
        f"chunk_size={DQ_CHUNK_SIZE}; "
        f"evidence_limit={DQ_EVIDENCE_LIMIT}."
    ),
    level="INFO",
)


try:
    for file_path in discovered_files:

        file_name = file_path.name

        log_event(
            message=(
                "DATA_QUALITY_FILE_START: "
                f"file={file_name}."
            ),
            level="INFO",
        )

        try:
            header_frame = pd.read_csv(
                file_path,
                nrows=0,
            )

            available_columns = list(
                header_frame.columns
            )

            missing_quality_columns = [
                column_name
                for column_name in QUALITY_REQUIRED_COLUMNS
                if column_name not in available_columns
            ]

            file_metrics = initialise_file_metrics(
                file_name=file_name,
                available_columns=available_columns,
                missing_columns=missing_quality_columns,
            )

            quality_metrics_by_file[
                file_name
            ] = file_metrics

            for missing_column in missing_quality_columns:
                add_quality_evidence(
                    metric_name="missing_quality_columns",
                    evidence_record={
                        "file_name": file_name,
                        "column": missing_column,
                    },
                )

            selected_columns = [
                column_name
                for column_name in QUALITY_REQUIRED_COLUMNS
                if column_name in available_columns
            ]

            if not selected_columns:
                raise ValueError(
                    "No configured quality-rule columns are "
                    "available in the file."
                )

            file_dtype_map = {
                column_name: "string"
                for column_name in selected_columns
                if column_name in KNOWN_TEXT_COLUMNS
            }

            rows_seen_before_chunk = 0

            chunk_iterator = pd.read_csv(
                file_path,
                usecols=selected_columns,
                dtype=file_dtype_map or None,
                chunksize=DQ_CHUNK_SIZE,
                low_memory=False,
            )

            for chunk in chunk_iterator:

                chunk_row_count = len(chunk)

                if chunk_row_count == 0:
                    continue

                source_row_numbers = pd.Series(
                    range(
                        rows_seen_before_chunk + 2,
                        rows_seen_before_chunk
                        + chunk_row_count
                        + 2,
                    ),
                    index=chunk.index,
                    dtype="int64",
                )

                rows_seen_before_chunk += chunk_row_count

                file_metrics["rows_scanned"] += (
                    chunk_row_count
                )

                file_metrics["chunks_processed"] += 1

                dataset_quality_metrics[
                    "total_rows_scanned"
                ] += chunk_row_count


                # ------------------------------
                # Null and blank metrics
                # ------------------------------

                for column_name in selected_columns:

                    column_series = chunk[column_name]

                    null_count = int(
                        column_series.isna().sum()
                    )

                    file_metrics["null_counts"][
                        column_name
                    ] += null_count

                    dataset_quality_metrics[
                        "null_counts"
                    ][column_name] += null_count

                    if column_name in KNOWN_TEXT_COLUMNS:
                        blank_mask = (
                            column_series.notna()
                            & column_series.astype(
                                "string"
                            ).str.strip().eq("")
                        )

                        blank_count = int(
                            blank_mask.sum()
                        )

                        file_metrics["blank_counts"][
                            column_name
                        ] += blank_count

                        dataset_quality_metrics[
                            "blank_counts"
                        ][column_name] += blank_count

                        if blank_count:
                            for row_index in chunk.index[
                                blank_mask
                            ][:DQ_EVIDENCE_LIMIT]:
                                add_quality_evidence(
                                    metric_name=(
                                        f"blank_{column_name}"
                                    ),
                                    evidence_record={
                                        "file_name": file_name,
                                        "source_row_number": int(
                                            source_row_numbers.loc[
                                                row_index
                                            ]
                                        ),
                                        "column": column_name,
                                    },
                                )


                # ------------------------------
                # Category frequencies
                # ------------------------------

                for category_column in CATEGORY_COLUMNS:

                    if category_column not in chunk:
                        continue

                    normalised_values = (
                        chunk[category_column]
                        .dropna()
                        .astype("string")
                        .str.strip()
                    )

                    value_counts = Counter(
                        normalised_values.tolist()
                    )

                    file_metrics["category_counts"][
                        category_column
                    ].update(value_counts)

                    dataset_quality_metrics[
                        "category_counts"
                    ][category_column].update(
                        value_counts
                    )


                # ------------------------------
                # Datetime and duration metrics
                # ------------------------------

                parsed_started_at = None
                parsed_ended_at = None

                if "started_at" in chunk:
                    raw_started_at = chunk[
                        "started_at"
                    ]

                    parsed_started_at = pd.to_datetime(
                        raw_started_at,
                        errors="coerce",
                    )

                    started_unparseable_mask = (
                        raw_started_at.notna()
                        & parsed_started_at.isna()
                    )

                    started_unparseable_count = int(
                        started_unparseable_mask.sum()
                    )

                    file_metrics["datetime_metrics"][
                        "started_at_unparseable"
                    ] += started_unparseable_count

                    dataset_quality_metrics[
                        "datetime_metrics"
                    ][
                        "started_at_unparseable"
                    ] += started_unparseable_count


                if "ended_at" in chunk:
                    raw_ended_at = chunk[
                        "ended_at"
                    ]

                    parsed_ended_at = pd.to_datetime(
                        raw_ended_at,
                        errors="coerce",
                    )

                    ended_unparseable_mask = (
                        raw_ended_at.notna()
                        & parsed_ended_at.isna()
                    )

                    ended_unparseable_count = int(
                        ended_unparseable_mask.sum()
                    )

                    file_metrics["datetime_metrics"][
                        "ended_at_unparseable"
                    ] += ended_unparseable_count

                    dataset_quality_metrics[
                        "datetime_metrics"
                    ][
                        "ended_at_unparseable"
                    ] += ended_unparseable_count


                if (
                    parsed_started_at is not None
                    and parsed_ended_at is not None
                ):
                    duration_seconds = (
                        parsed_ended_at
                        - parsed_started_at
                    ).dt.total_seconds()

                    non_positive_mask = (
                        duration_seconds.notna()
                        & duration_seconds.le(0)
                    )

                    over_24_hours_mask = (
                        duration_seconds.notna()
                        & duration_seconds.gt(
                            24 * 60 * 60
                        )
                    )

                    non_positive_count = int(
                        non_positive_mask.sum()
                    )

                    over_24_hours_count = int(
                        over_24_hours_mask.sum()
                    )

                    file_metrics["datetime_metrics"][
                        "non_positive_duration"
                    ] += non_positive_count

                    file_metrics["datetime_metrics"][
                        "duration_over_24_hours"
                    ] += over_24_hours_count

                    dataset_quality_metrics[
                        "datetime_metrics"
                    ][
                        "non_positive_duration"
                    ] += non_positive_count

                    dataset_quality_metrics[
                        "datetime_metrics"
                    ][
                        "duration_over_24_hours"
                    ] += over_24_hours_count

                    update_duration_aggregates(
                        metrics=file_metrics[
                            "datetime_metrics"
                        ],
                        duration_seconds=duration_seconds,
                    )

                    update_duration_aggregates(
                        metrics=dataset_quality_metrics[
                            "datetime_metrics"
                        ],
                        duration_seconds=duration_seconds,
                    )

                    for row_index in chunk.index[
                        non_positive_mask
                    ][:DQ_EVIDENCE_LIMIT]:
                        add_quality_evidence(
                            metric_name=(
                                "non_positive_duration"
                            ),
                            evidence_record={
                                "file_name": file_name,
                                "source_row_number": int(
                                    source_row_numbers.loc[
                                        row_index
                                    ]
                                ),
                                "ride_id": (
                                    None
                                    if "ride_id" not in chunk
                                    else chunk.at[
                                        row_index,
                                        "ride_id",
                                    ]
                                ),
                                "started_at": str(
                                    chunk.at[
                                        row_index,
                                        "started_at",
                                    ]
                                ),
                                "ended_at": str(
                                    chunk.at[
                                        row_index,
                                        "ended_at",
                                    ]
                                ),
                            },
                        )


                # ------------------------------
                # Station completeness metrics
                # ------------------------------

                for station_column in STATION_COLUMNS:

                    if station_column not in chunk:
                        continue

                    station_series = chunk[
                        station_column
                    ]

                    missing_station_mask = (
                        station_series.isna()
                        | station_series.astype(
                            "string"
                        ).str.strip().eq("")
                    )

                    missing_station_count = int(
                        missing_station_mask.sum()
                    )

                    file_metrics[
                        "station_missing_counts"
                    ][station_column] += (
                        missing_station_count
                    )

                    dataset_quality_metrics[
                        "station_missing_counts"
                    ][station_column] += (
                        missing_station_count
                    )


                # ------------------------------
                # Coordinate metrics
                # ------------------------------

                for coordinate_column, (
                    minimum_value,
                    maximum_value,
                ) in COORDINATE_LIMITS.items():

                    if coordinate_column not in chunk:
                        continue

                    raw_coordinate = chunk[
                        coordinate_column
                    ]

                    numeric_coordinate = pd.to_numeric(
                        raw_coordinate,
                        errors="coerce",
                    )

                    null_mask = raw_coordinate.isna()

                    non_numeric_mask = (
                        raw_coordinate.notna()
                        & numeric_coordinate.isna()
                    )

                    below_minimum_mask = (
                        numeric_coordinate.notna()
                        & numeric_coordinate.lt(
                            minimum_value
                        )
                    )

                    above_maximum_mask = (
                        numeric_coordinate.notna()
                        & numeric_coordinate.gt(
                            maximum_value
                        )
                    )

                    coordinate_counts = {
                        "null_count": int(
                            null_mask.sum()
                        ),
                        "non_numeric_count": int(
                            non_numeric_mask.sum()
                        ),
                        "below_minimum_count": int(
                            below_minimum_mask.sum()
                        ),
                        "above_maximum_count": int(
                            above_maximum_mask.sum()
                        ),
                    }

                    for metric_name, metric_count in (
                        coordinate_counts.items()
                    ):
                        file_metrics[
                            "coordinate_metrics"
                        ][coordinate_column][
                            metric_name
                        ] += metric_count

                        dataset_quality_metrics[
                            "coordinate_metrics"
                        ][coordinate_column][
                            metric_name
                        ] += metric_count


                # ------------------------------
                # Disk-backed ride ID tracking
                # ------------------------------

                if "ride_id" in chunk:
                    insert_ride_id_occurrences(
                        ride_ids=chunk["ride_id"],
                        source_file=file_name,
                        source_row_numbers=(
                            source_row_numbers
                        ),
                    )

                duplicate_connection.commit()


            file_metrics["status"] = "COMPLETE"

            file_metrics[
                "scan_completed_at"
            ] = datetime.now(
                MELBOURNE_TIMEZONE
            ).isoformat(timespec="seconds")

            dataset_quality_metrics[
                "files_scanned_successfully"
            ] += 1

            log_event(
                message=(
                    "DATA_QUALITY_FILE_COMPLETE: "
                    f"file={file_name}; "
                    f"rows={file_metrics['rows_scanned']}; "
                    f"chunks={file_metrics['chunks_processed']}."
                ),
                level="INFO",
            )


        except (
            pd.errors.EmptyDataError,
            pd.errors.ParserError,
            UnicodeDecodeError,
            OSError,
            ValueError,
            TypeError,
        ) as error:

            failure_record = {
                "file_name": file_name,
                "failed_at": datetime.now(
                    MELBOURNE_TIMEZONE
                ).isoformat(timespec="seconds"),
                "error_type": type(error).__name__,
                "error_message": str(error),
            }

            quality_scan_failures.append(
                failure_record
            )

            dataset_quality_metrics[
                "files_failed"
            ] += 1

            if file_name in quality_metrics_by_file:
                quality_metrics_by_file[
                    file_name
                ]["status"] = "FAILED"

                quality_metrics_by_file[
                    file_name
                ]["failure"] = deepcopy(
                    failure_record
                )

            log_event(
                message=(
                    "DATA_QUALITY_FILE_FAILED: "
                    f"file={file_name}; "
                    f"error_type={type(error).__name__}; "
                    f"error={error}"
                ),
                level="ERROR",
            )


    # ------------------------------------------
    # Build duplicate indexes
    # ------------------------------------------

    duplicate_cursor.execute(
        """
        CREATE INDEX idx_ride_id
        ON ride_id_occurrences (ride_id)
        """
    )

    duplicate_cursor.execute(
        """
        CREATE INDEX idx_ride_id_file
        ON ride_id_occurrences (
            ride_id,
            source_file
        )
        """
    )

    duplicate_connection.commit()


    # ------------------------------------------
    # Dataset-level duplicate counts
    # ------------------------------------------

    duplicate_cursor.execute(
        """
        SELECT
            COUNT(*) AS duplicate_ride_id_count,
            COALESCE(SUM(occurrence_count), 0)
                AS duplicate_rows_affected
        FROM (
            SELECT
                ride_id,
                COUNT(*) AS occurrence_count
            FROM ride_id_occurrences
            GROUP BY ride_id
            HAVING COUNT(*) > 1
        )
        """
    )

    (
        duplicate_ride_id_count,
        duplicate_rows_affected,
    ) = duplicate_cursor.fetchone()

    dataset_quality_metrics["duplicates"][
        "duplicate_ride_id_count"
    ] = int(duplicate_ride_id_count or 0)

    dataset_quality_metrics["duplicates"][
        "duplicate_rows_affected"
    ] = int(duplicate_rows_affected or 0)


    # ------------------------------------------
    # Cross-file duplicate count
    # ------------------------------------------

    duplicate_cursor.execute(
        """
        SELECT COUNT(*)
        FROM (
            SELECT ride_id
            FROM ride_id_occurrences
            GROUP BY ride_id
            HAVING COUNT(DISTINCT source_file) > 1
        )
        """
    )

    cross_file_duplicate_count = (
        duplicate_cursor.fetchone()[0]
    )

    dataset_quality_metrics["duplicates"][
        "cross_file_duplicate_ride_id_count"
    ] = int(cross_file_duplicate_count or 0)


    # ------------------------------------------
    # Within-file duplicate metrics
    # ------------------------------------------

    duplicate_cursor.execute(
        """
        SELECT
            source_file,
            COUNT(*) AS duplicate_id_count,
            COALESCE(SUM(occurrence_count), 0)
                AS duplicate_rows_affected
        FROM (
            SELECT
                source_file,
                ride_id,
                COUNT(*) AS occurrence_count
            FROM ride_id_occurrences
            GROUP BY source_file, ride_id
            HAVING COUNT(*) > 1
        )
        GROUP BY source_file
        """
    )

    within_file_results = (
        duplicate_cursor.fetchall()
    )

    total_within_file_duplicate_ids = 0

    for (
        source_file,
        duplicate_id_count,
        duplicate_rows_affected,
    ) in within_file_results:

        total_within_file_duplicate_ids += int(
            duplicate_id_count
        )

        if source_file in quality_metrics_by_file:
            quality_metrics_by_file[
                source_file
            ]["duplicates"][
                "duplicate_ride_id_count"
            ] = int(duplicate_id_count)

            quality_metrics_by_file[
                source_file
            ]["duplicates"][
                "duplicate_rows_affected"
            ] = int(duplicate_rows_affected)

    dataset_quality_metrics["duplicates"][
        "within_file_duplicate_ride_id_count"
    ] = total_within_file_duplicate_ids


    # ------------------------------------------
    # Bounded duplicate evidence
    # ------------------------------------------

    duplicate_cursor.execute(
        """
        SELECT
            occurrence.ride_id,
            occurrence.source_file,
            occurrence.source_row_number
        FROM ride_id_occurrences AS occurrence
        INNER JOIN (
            SELECT ride_id
            FROM ride_id_occurrences
            GROUP BY ride_id
            HAVING COUNT(*) > 1
        ) AS duplicate_ids
            ON occurrence.ride_id = duplicate_ids.ride_id
        ORDER BY
            occurrence.ride_id,
            occurrence.source_file,
            occurrence.source_row_number
        LIMIT ?
        """,
        (DQ_EVIDENCE_LIMIT,),
    )

    for (
        ride_id,
        source_file,
        source_row_number,
    ) in duplicate_cursor.fetchall():

        add_quality_evidence(
            metric_name="duplicate_ride_id",
            evidence_record={
                "ride_id": ride_id,
                "file_name": source_file,
                "source_row_number": int(
                    source_row_number
                ),
            },
        )


finally:
    duplicate_cursor.close()
    duplicate_connection.close()

    if DUPLICATE_TRACKER_PATH.exists():
        DUPLICATE_TRACKER_PATH.unlink()


# ------------------------------------------
# Finalise calculated duration averages
# ------------------------------------------

for file_metrics in quality_metrics_by_file.values():

    duration_metrics = file_metrics[
        "datetime_metrics"
    ]

    valid_duration_count = duration_metrics[
        "valid_duration_count"
    ]

    duration_metrics["duration_average_seconds"] = (
        duration_metrics["duration_total_seconds"]
        / valid_duration_count
        if valid_duration_count
        else None
    )


dataset_duration_metrics = dataset_quality_metrics[
    "datetime_metrics"
]

dataset_valid_duration_count = (
    dataset_duration_metrics[
        "valid_duration_count"
    ]
)

dataset_duration_metrics[
    "duration_average_seconds"
] = (
    dataset_duration_metrics[
        "duration_total_seconds"
    ] / dataset_valid_duration_count
    if dataset_valid_duration_count
    else None
)


# ------------------------------------------
# Convert Counters into standard dictionaries
# ------------------------------------------

quality_metrics_by_file = counter_to_dict(
    quality_metrics_by_file
)

dataset_quality_metrics = counter_to_dict(
    dataset_quality_metrics
)


dataset_quality_metrics[
    "completed_at"
] = datetime.now(
    MELBOURNE_TIMEZONE
).isoformat(timespec="seconds")


# ------------------------------------------
# Record metric-generation completion
# ------------------------------------------

log_event(
    message=(
        "DATA_QUALITY_METRIC_SCAN_COMPLETE: "
        f"files_scanned="
        f"{dataset_quality_metrics['files_scanned_successfully']}; "
        f"files_failed="
        f"{dataset_quality_metrics['files_failed']}; "
        f"rows_scanned="
        f"{dataset_quality_metrics['total_rows_scanned']}; "
        f"duplicate_ride_ids="
        f"{dataset_quality_metrics['duplicates']['duplicate_ride_id_count']}."
    ),
    level="INFO",
)


# ------------------------------------------
# Display metric summary
# ------------------------------------------

print("\n========== Data Quality Metric Summary ==========\n")

print(
    f"Scan mode                   : "
    f"{dataset_quality_metrics['scan_mode']}"
)
print(
    f"Chunk size                  : "
    f"{dataset_quality_metrics['chunk_size']:,}"
)
print(
    f"Files discovered            : "
    f"{dataset_quality_metrics['files_discovered']}"
)
print(
    f"Files scanned successfully  : "
    f"{dataset_quality_metrics['files_scanned_successfully']}"
)
print(
    f"Files failed                : "
    f"{dataset_quality_metrics['files_failed']}"
)
print(
    f"Total rows scanned          : "
    f"{dataset_quality_metrics['total_rows_scanned']:,}"
)
print(
    f"Duplicate ride IDs          : "
    f"{dataset_quality_metrics['duplicates']['duplicate_ride_id_count']:,}"
)
print(
    f"Duplicate rows affected     : "
    f"{dataset_quality_metrics['duplicates']['duplicate_rows_affected']:,}"
)
print(
    f"Cross-file duplicate IDs    : "
    f"{dataset_quality_metrics['duplicates']['cross_file_duplicate_ride_id_count']:,}"
)
print(
    f"Evidence categories retained: "
    f"{len(quality_metric_evidence)}"
)

print("\nFiles:")

for file_name, file_metrics in (
    quality_metrics_by_file.items()
):
    print(
        f"  {file_name:<40} "
        f"status={file_metrics['status']:<10} "
        f"rows={file_metrics['rows_scanned']:,}"
    )

if quality_scan_failures:
    print("\nScan failures:")

    for failure in quality_scan_failures:
        print(
            f"  - {failure['file_name']}: "
            f"{failure['error_type']} — "
            f"{failure['error_message']}"
        )

print("\n" + "=" * 75)
print("✓ Full source files were scanned in bounded-memory chunks.")
print("✓ Aggregate quality metrics are available in memory.")
print("✓ Cross-file duplicate tracking completed.")
print("✓ Evidence collection remained bounded.")
print("✓ Results are ready for Part 3.3 assessment.")
print("=" * 75)